In [1]:
# Install the project dependencies.
%pip install -q -r ../../requirements.txt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

# The run that the paper reports. Do not change without updating CLAUDE.md.
RUN = ROOT / "runs/estimation/2026-06-13_22-11-06_global_IC_country_trends"
FIGURES = ROOT / "paper/Figures"

print("run    ", RUN.name)
print("exists ", RUN.exists())


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml

from utils import create_pred_input, add_histogram
from hdf5_min import read_weights

cfg = yaml.safe_load((RUN / ".hydra/config.yaml").read_text(encoding="utf-8"))["instance"]
cfg = {**cfg["optim"], **cfg["model"], **cfg["server"]}

for k in ("target_mode", "within_projection", "country_trends", "data_source"):
    print(f"{k:20} {cfg.get(k)}")


In [ ]:
# Observed climate, used for the standardisation and for the density floor.
data = pd.read_excel(ROOT / "data/MainData.xlsx")

mean_temp = np.nanmean(data["TempPopWeight"])
std_temp = np.nanstd(data["TempPopWeight"])
mean_precip = np.nanmean(data["PrecipPopWeight"])
std_precip = np.nanstd(data["PrecipPopWeight"])

pred_input, T, P = create_pred_input(
    mc=False, mean_T=mean_temp, std_T=std_temp,
    mean_P=mean_precip, std_P=std_precip, precip_capped=True)

print("grid", T.shape, " mean T %.2f  mean P %.0f mm" % (mean_temp, mean_precip))


In [ ]:
# The ten architectures with the lowest BIC.
results = dict(np.load(RUN / "results.npy", allow_pickle=True).item())
results = {k: v for k, v in results.items() if v is not None}

n_models = 10
top_models = sorted(results, key=lambda node: results[node][0])[:n_models]

for rank, node in enumerate(top_models, 1):
    print(f"{rank:>2}. {str(node):<14} BIC {float(results[node][0]):>12.1f}")


In [ ]:
# Forward pass straight from the saved weights.
# This run uses within_projection, so the saved graph is the climate net alone:
# hidden layers with a bias, then an output layer without one.

def swish(z):
    return z / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))

def forward(w, X):
    idx = sorted({int(k.split("/")[1].split("_")[1]) if "_" in k.split("/")[1] else 0
                  for k in w if k.startswith("layers")})
    h = X
    for i in idx:
        b = f"layers/dense{'' if i == 0 else f'_{i}'}/vars"
        h = h @ np.asarray(w[f"{b}/0"], float)
        if f"{b}/1" in w:
            h = swish(h + np.asarray(w[f"{b}/1"], float))
    return h[..., 0]

# Centre each surface at the sample mean climate. The inputs are standardised,
# so the all-zeros input is that mean. An additive constant is not separately
# identified from the country fixed effects, so this only fixes the level.
ORIGIN = np.zeros((1,) * (pred_input.ndim - 1) + (pred_input.shape[-1],))

def surface(node):
    w = read_weights(str(RUN / "parameters" / f"{node}.weights.h5"))
    Z = forward(w, pred_input).reshape(T.shape)
    return Z - float(np.ravel(forward(w, ORIGIN))[0])

surfaces = {node: surface(node) for node in top_models
            if (RUN / "parameters" / f"{node}.weights.h5").exists()}
z = np.mean(list(surfaces.values()), axis=0)

for node, Z in surfaces.items():
    print(f"{str(node):>12}  {Z.min():+.4f} to {Z.max():+.4f}")
print(f"{'mean':>12}  {z.min():+.4f} to {z.max():+.4f}  (n={len(surfaces)})")


In [ ]:
mean_surface = go.Surface(
    x=T, y=P / 1000, z=z,
    colorscale="Cividis", opacity=0.85, showscale=False)

fig = go.Figure(data=[mean_surface])
fig.update_layout(
    autosize=True,
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis_title="Temperature (°C)",
        yaxis_title="Precipitation (m)",
        zaxis=dict(title=dict(text="Δ ln(GDP)"), range=[-0.3, 0.3]),
        camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589))),
    font=dict(size=10))

# Observed density on the floor, so a reader can see which parts of the
# surface are supported by data.
fig = add_histogram(fig, data, legend=False, precip_capped=True)
fig.show()


In [ ]:
from pypdf import PdfReader, PdfWriter

# Crop to the same page box as the other surface figures.
CROP_REFERENCE = FIGURES / "Global/global_top10surfaces_CT_fit.pdf"
CROP_ADJUST_PT = {"left": -40.0, "bottom": 0.0, "right": 30.0, "top": 30.0}

def crop_pdf(path, reference, adjust):
    box = PdfReader(str(reference)).pages[0].cropbox
    left = float(box.left) + adjust["left"]
    bottom = float(box.bottom) + adjust["bottom"]
    right = float(box.right) + adjust["right"]
    top = float(box.top) - adjust["top"]
    reader, writer = PdfReader(str(path)), PdfWriter()
    for page in reader.pages:
        page.cropbox.lower_left = (left, bottom)
        page.cropbox.upper_right = (right, top)
        page.mediabox.lower_left = (left, bottom)
        page.mediabox.upper_right = (right, top)
        writer.add_page(page)
    with open(path, "wb") as fh:
        writer.write(fh)
    return left, bottom, right, top

pdf = FIGURES / "Global/global_mean_CT_fit.pdf"
pdf.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(pdf, width=900, height=700, scale=2)
print("crop", crop_pdf(pdf, CROP_REFERENCE, CROP_ADJUST_PT))
print("->", pdf)
